In [0]:
jdbc_url = "jdbc:sqlserver://lfsql16611.database.windows.net:1433;database=lfdemo;encrypt=true;trustServerCertificate=false"

connection_props = {
    "user": "sqladminuser",
    "password": "StrongP@ssw0rd!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


In [0]:
df_fact = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.FACT_TRANSACTION",
    properties=connection_props
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS learning_lab_catalog.control.fact_watermark (
    table_name STRING,
    last_processed_ts TIMESTAMP
);


In [0]:
%sql
INSERT INTO learning_lab_catalog.control.fact_watermark
VALUES ('FACT_TRANSACTION', '1900-01-01');


In [0]:
watermark_df = spark.sql("""
SELECT last_processed_ts
FROM learning_lab_catalog.control.fact_watermark
WHERE table_name = 'FACT_TRANSACTION'
""")

last_ts = watermark_df.collect()[0][0]
print("Last processed timestamp:", last_ts)


In [0]:
incremental_query = f"""
(
 SELECT *
 FROM dbo.FACT_TRANSACTION
 WHERE updated_at > '{last_ts}'
) AS src
"""

source_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table=incremental_query,
        properties=connection_props
    )
)

In [0]:
%sql
select*from learning_lab_catalog.control.fact_watermark